In [ ]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

# Modeling
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

In [21]:
# Import CSV and convert to pandas dataframe
data_path = "/Users/toripalecek/Documents/metafer_usage/metafer_usage/cleaned_data/cleaned_metafer_usage.csv"
data = pd.read_csv(data_path)
df = pd.DataFrame(data)
df.head()

,case_number,capture_date,scans,metafer,date,time,time_bin,month,weekday,weekday_index
0,26-684_MDSDF,2026-01-02 15:17:00,6,meta 2,2026-01-02,15:17:00,15:00:00,1,Friday,4
1,26-697_AMLFA,2026-01-02 13:47:00,10,meta 2,2026-01-02,13:47:00,13:30:00,1,Friday,4
2,26-698_AMLFA,2026-01-02 13:52:00,10,meta 7,2026-01-02,13:52:00,13:30:00,1,Friday,4
3,26-1081_EOSMF,2026-01-03 13:45:00,1,meta 1,2026-01-03,13:45:00,13:30:00,1,Saturday,5
4,26-1884_AMLFA,2026-01-02 13:51:00,4,meta 3,2026-01-02,13:51:00,13:30:00,1,Friday,4


In [ ]:
# Convert time_bin to datetime and calculate operational minutes since shifts cross midnight (3:00 AM)
#time = pd.to_datetime(df['time_bin'])

#df['operational_minutes'] = (
#    time.dt.hour * 60 + time.dt.minute - 180
#) % (24*60)
#df.head()

/var/folders/rr/bmtx3y515yq966zhdj0_1rhw0000gn/T/ipykernel_26209/91763925.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time = pd.to_datetime(df['time_bin'])


,case_number,capture_date,scans,metafer,date,time,time_bin,month,weekday,weekday_index,operational_minutes
0,26-684_MDSDF,2026-01-02 15:17:00,6,meta 2,2026-01-02,15:17:00,15:00:00,1,Friday,4,720
1,26-697_AMLFA,2026-01-02 13:47:00,10,meta 2,2026-01-02,13:47:00,13:30:00,1,Friday,4,630
2,26-698_AMLFA,2026-01-02 13:52:00,10,meta 7,2026-01-02,13:52:00,13:30:00,1,Friday,4,630
3,26-1081_EOSMF,2026-01-03 13:45:00,1,meta 1,2026-01-03,13:45:00,13:30:00,1,Saturday,5,630
4,26-1884_AMLFA,2026-01-02 13:51:00,4,meta 3,2026-01-02,13:51:00,13:30:00,1,Friday,4,630


In [22]:
# Create a custom sklearn transformer for the time column. Convert into minutes with consideration for when shift changes are.
class TimeToOperationalMinutes(BaseEstimator, TransformerMixin):
    def __init__(self, start_hour=3):
        self.start_hour = start_hour

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        time = pd.to_datetime(X.iloc[:, 0])

        operational_minutes = (
            time.dt.hour * 60 
            + time.dt.minute 
            - self.start_hour * 60
        ) % (24 * 60)

        return operational_minutes.to_numpy().reshape(-1, 1)

In [23]:
# Drop unnecessary columns for clustering
drop_cols = ['case_number','capture_date','time','month','weekday_index','date']
df_drop = df.drop(drop_cols, axis=1)
df_drop.head()

,scans,metafer,time_bin,weekday
0,6,meta 2,15:00:00,Friday
1,10,meta 2,13:30:00,Friday
2,10,meta 7,13:30:00,Friday
3,1,meta 1,13:30:00,Saturday
4,4,meta 3,13:30:00,Friday


In [24]:
df_drop.shape

(6571, 4)

In [25]:
# Create a column transformer to preprocess the data for clustering
num_features = ['scans']
cat_features = ['metafer','weekday']
time_feature = ['time_bin']

numberic_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

time_transformer = Pipeline(
    steps=[
        ("convert_time", TimeToOperationalMinutes()),
        ("scale_time", StandardScaler())
    ]
)
preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("Time", time_transformer, time_feature),
        ("StandardScaler", numberic_transformer, num_features)])

In [26]:
df_drop = preprocessor.fit_transform(df_drop)

/var/folders/rr/bmtx3y515yq966zhdj0_1rhw0000gn/T/ipykernel_31777/2078213797.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time = pd.to_datetime(X.iloc[:, 0])


In [27]:
# Check the shape of the transformed dataframe
df_drop.shape

(6571, 14)

In [ ]:
# Model Evaluation Metric
mask = labels != -1  # Exclude noise points
score = silhouette_score(df_drop[mask], labels[mask])

inertia = []

for k in range(2,10):
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_processed)
    inertia.append(km.inertia_)


In [ ]:
models = {
    "KMeans": KMeans(),
    "DBSCAN": DBSCAN()
}

params = {
    "KMeans": {
        "n_clusters": [2, 3, 4, 5, 6, 8, 10],
        "init": ["k-means++", "random"],
        "n_init": [10, 20, 30],
        "max_iter": [300, 500, 1000]
    },
    "DBSCAN": {
        "eps": [0.1, 0.5, 1.0],
        "min_samples": [3, 5, 10],
        "metric": ["euclidean", "manhattan"]
    }
}

model_list = []